# Experiment 2: Repetitions


In [ ]:
import os
import pandas as pd
import json
from os.path import join
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score
from collections import defaultdict
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import scipy.stats as st
import matplotlib.cm as cm
import matplotlib.colorbar as colorbar
from statsmodels.stats.contingency_tables import cochrans_q, mcnemar, Table
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint
import plotly.graph_objects as go


import itertools

In [ ]:
# ESAP questions with images
q_w_images = [3, 4, 6, 32, 35, 54, 68, 88]

cases = np.delete(np.arange(100), [0] + q_w_images) # valid questions
n_cases = cases.size

# Available options
options = ['A', 'B', 'C', 'D', 'E']
# even for the experiments without letter, manual annotator saved the option using letter.

path_to_results = "./../results"

model_file_name_dict = {'huatuo-o1': 'HuatuoGPT-o1',
                        'diabetica-o1': 'Diabetica-o1',
                        'diabetica-7B': 'Diabetica-7B',
                        'meditron3-8B': 'Meditron3-8B'}

In [ ]:
def evaluate_response_distrib(filepath):
    """ 
    Given the excel file, this function parses and standardize the file entries.
    Returns a counter in the form of a dictionary.
    """
    df_ = pd.read_excel(filepath, header=None, index_col=0)
    
    # everything capital
    df_ = df_.map(lambda x: x.upper() if isinstance(x, str) else x) 
    
    # remove NaN values - this remove extra-columns
    df_ = df_.dropna(axis=1, how='all') 
    if not np.array_equal(df_.index, cases): # we keep all 91 cases
        raise ValueError("We are removing a valid case by mistake! Check Excel file")

    # unique annotations
    # print(df_.values)
    print(np.unique(df_.values))
    unique_elements = np.unique(df_.values) 

    # multiple options selected
    # are indicated as option1-option2: there is a dash
    bm_multiple = ["-" in v_ for v_ in unique_elements] 
    
    bm_hall_none = [True if (v_ == "HALL" or v_ == 'NONE') # hallucination or no response
                    else False for v_ in unique_elements]
    
    bm_others = ~np.logical_or(bm_multiple, bm_hall_none) # correct answers / check (temporary)
    
    idx_multiple = np.where(bm_multiple)[0] # indexes
    idx_hall_none = np.where(bm_hall_none)[0]
        
    dct_counter = {}
    all_ = 0 # to make sure we did not miss any response (91 or multiple for 10 rep)

    # we are creating a new label for multi and hall/none
    if len(idx_multiple) > 0:
        count = 0
        for v_ in unique_elements[idx_multiple]:
            count += (df_.values == v_).sum()
        all_ = count
        dct_counter["MULTI"] = count
            
    if len(idx_hall_none) > 0:
        count = 0
        for v_ in unique_elements[idx_hall_none]:
            count += (df_.values == v_).sum()
        all_ += count
        dct_counter["HALL / NONE"] = count
       
    # we are using the old label for everything else
    for element in unique_elements[bm_others]:
        count = (df_.values == element).sum()
        dct_counter[element] = count
        all_ += count
            
    list_add = ["CHECK", "MULTI", "HALL / NONE", "A", "B", "C", "D", "E"]
    for el_ in list_add:
        if el_ not in dct_counter.keys():
            dct_counter[el_] = 0
    dct_counter["TOTAL"] = all_
        
    return dct_counter

In [ ]:
model_paths_token = {'meditron3-8B-prompt1': 'meditron3-8B_promptID_001_output_4.xlsx',
                     'huatuo-o1-prompt1': 'huatuo-o1_promptID_001_output_9.xlsx',
                     'huatuo-o1-prompt2': 'huatuo-o1_promptID_002_output_11.xlsx',
                     'diabetica-o1-prompt1': 'diabetica-o1_promptID_001_output_2.xlsx',
                     'diabetica-o1-prompt2': 'diabetica-o1_promptID_002_output_3.xlsx',
                     'diabetica-7B-prompt1': 'diabetica-7B_promptID_001_output_3.xlsx',
                     'diabetica-7B-prompt2': 'diabetica-7B_promptID_002_output_4.xlsx',
                     'medfound7B-prompt1': 'medfound7B_promptID_001_output_2.xlsx',
                     'medfound7B-prompt2': 'medfound7B_promptID_002_output_3.xlsx',
                     'clinical-chatgpt-prompt1': 'clinical-chatgpt_promptID_001_output_2.xlsx',
                     'clinical-chatgpt-prompt2': 'clinical-chatgpt_promptID_002_output_3.xlsx',
                     'bloom-7b-prompt1': 'bloom-7B_promptID_001_output_0.xlsx',
                     'bloom-7b-prompt2': 'bloom-7B_promptID_002_output_1.xlsx',
                     'llama31-prompt1': 'llama31_promptID_001_output_6.xlsx',
                     'llama31-prompt2': 'llama31_promptID_002_output_7.xlsx',
                     'qwen2-7b-prompt1': 'qwen2-7b_promptID_001_output_0.xlsx',
                     'qwen2-7b-prompt2': 'qwen2-7b_promptID_002_output_1.xlsx',
                     'huatuo-o1-lms-prompt1': 'huatuo-o1-lms-prompt001.xlsx',
                     'gpt-oss-20b-prompt1': 'gpt-oss-20b-lms-prompt001.xlsx'
                    }

model_paths_no_token = {'huatuo-o1': 'huatuo-o1_promptID_001_output_47.xlsx',
                        'diabetica-o1': 'diabetica-o1_promptID_001_output_40.xlsx',
                        'diabetica-7B': 'diabetica-7B_promptID_001_output_47.xlsx',
                        'meditron3-8B': 'meditron3-8B_promptID_001_output_39.xlsx',
                        'clinical-chatgpt': 'clinical-chatgpt_promptID_001_output_4.xlsx',
                        'medfound7B': 'medfound7B_promptID_001_output_4.xlsx',
                        'bloom-7b': 'bloom-7b_promptID_001_output_2.xlsx',
                        'llama31': 'llama31_promptID_001_output_8.xlsx',
                        'qwen2-7b': 'qwen2-7b_promptID_001_output_2.xlsx'
                       }

In [ ]:
def counter_extensive_dfs(result_folder, dct_models, letter=True):
    """ 
    Used in experiment 1 to a) keep track of distribution of responses
    b) build dataframe with extensive results.
    
    Variables: 
    result_folder: str, folder containing model's output
    dct_models: str, excel file 
    letter: bool, if True, ESAP question, otherwise, letter removed from option
    
    Returns:
    df_counter: pd.DataFrame, distribution of models responses
    df_all: pd.DataFrame extensive (91 outputs)
    """
    list_df = []
    index = []
    counting_outputs = []

    for model_, file_ in dct_models.items():
        if letter:
            filepath = join(path_to_results, model_.split("-prompt")[0], file_)
        else: 
            filepath = join(path_to_results, "NO_LETTERS", file_)
            
        df_ = pd.read_excel(filepath, header=None, index_col=0)
        df_ = df_[1] # we keep first column, others are empty
        list_df.append(df_)
        
        dct_counter = evaluate_response_distrib(filepath)
        counting_outputs.append([dct_counter[k] for k in sorted(dct_counter.keys())])

    # this is for the counter
    columns_name = sorted(dct_counter.keys())
    
    list_model_name = [m_ if letter else f"{m_}_no_letter" for m_ in dct_models.keys()]

    df_counter = pd.DataFrame(data=counting_outputs, 
                              index=list_model_name,
                              columns=columns_name)

    # this is for the Df with 91 responses
    df_all = pd.concat(list_df, axis=1)
    df_all.columns = list_model_name
    
    return df_counter, df_all

In [ ]:
df_counter_prompt_A_B, df_all = counter_extensive_dfs(path_to_results, model_paths_token)

In [ ]:
df_counter_no_let, df_all_no_let = counter_extensive_dfs(path_to_results, model_paths_no_token, letter=False)

In [ ]:
# concatenate all results from experiment 1
df_all = pd.concat([df_all, df_all_no_let], axis=1)

In [ ]:
with open("./../inference/Ped-ESAP.json", "r") as f:
    content = json.load(f)
    
responses = []
labs = []
table = []
for k, v in content.items():
    if len(v['answer']) > 0:
        responses.append(v['answer'][0])
        labs.append(v["labs"]=='Yes')
        table.append(v["table"]=='Yes')
    else:
        responses.append(None)
        labs.append(None)
        table.append(None)
        
ground_truth = pd.DataFrame(data=np.array(responses[:-1]).reshape(-1,1), index=np.arange(1,100),
                           columns=["truth"])
ground_truth = ground_truth.drop(q_w_images)

meta_labs_table = np.hstack((np.array(labs).reshape(-1,1), np.array(table).reshape(-1,1)))
meta_labs_tab_df = pd.DataFrame(data=meta_labs_table[:-1, :], index=np.arange(1,100),
                    columns=["labs", "table"])
meta_labs_tab_df = meta_labs_tab_df.drop(q_w_images)

In [ ]:
# run only once, otherwise we get multiple "truth" columns
df_all = pd.concat([df_all, ground_truth], axis=1)

In [ ]:
# here we keep the value in entry if is one among A, B, C, D, or E
# we report as NaN otherwise
for col in df_all.columns[:-1]:
    df_all[col] = df_all[col].where(df_all[col].isin(options), np.nan)

In [ ]:
bm_valid = ~df_all.isna()

In [ ]:
accuracy_df = df_all[df_all.columns[:-1]].eq(df_all['truth'], axis=0)

accuracy_df.head()

In [ ]:
accuracy_scores = accuracy_df.sum()

In [ ]:
models_col = df_all.columns[:-1] # we exclude the truth column

In [ ]:
results = pd.concat([accuracy_scores[models_col], bm_valid[models_col].sum()], axis=1)
results.columns = ['# correct', '# interpretable']

In [ ]:
os.makedirs("tables", exist_ok=True)
os.makedirs("plots", exist_ok=True)


In [ ]:
model_list = ['HuatuoGPT-o1-8B', 'Meditron3-8B', 'Llama3.1-8B Instruct', 'Diabetica-o1', 'Diabetica-7B', 'Qwen2-7B Instruct']
model_file = ['huatuo-o1', 'meditron3-8B', 'llama31', 'diabetica-o1', 'diabetica-7B', 'qwen2-7b']
temperature_list = [0.3, 0.6, 1.]
path_repetitions = f"{path_to_results}/REPETITIONS"

The following comparison matrix contains the number of times the LLMs select the correct option. 

``rep_df`` is a Excel with 91 rows (cases) and 10 columns (repetitions).

We repeat this across the three temperatures tested.

In [ ]:
def max_frequency_string(arr, identify_majority=True):
    """ This code returns nan if there is no majorty selected (e.g., two options with same
    frequency). Otherwise, option and how many times it was selected. """
    # with identify mayority=True, we count how many times majority was selected
    # with identify_majority False, we do not care about majority, we just want the count of the max selection
    if not arr.size:
        return None, 0
    unique_elements, counts = np.unique(arr, return_counts=True)
    max_indexes = counts == np.max(counts)
    if np.sum(max_indexes) > 1:
        if identify_majority:
            return np.nan, np.nan # returns not a number if we do not have a majority
        else:
            return np.nan, np.max(counts)
    # elif unique_elements[np.argmax(counts)] not in options:
    #     return np.nan, np.nan # if max option is none
    else:
        return unique_elements[np.argmax(counts)], np.max(counts)

Counting outputs: There are some cases where models does not make a selection, we report these values as well.

In [ ]:
# what is the accuracy for each run, model, temperature
bm_across_models_and_t = []
comparison_mat_all = []
most_sel_list_and_t = []
count_sel_list_and_t = []

for t_ in temperature_list:
    bm_across_models = []
    comparison_mat = []
    most_sel_list = []
    count_sel_list = []

    for m_ in model_file: 
        # for each model and temperature we load the excel and retrieve information
        rep_df = pd.read_excel(f"{path_repetitions}/T={t_}/{m_}/{m_}_promptID_001_repetitions.xlsx",
                               header=None, index_col=0)
        comparison = rep_df.eq(df_all['truth'], axis=0)
        
        most_sel = []
        count_sel = []
        for id_row, row in enumerate(rep_df.values):
            row = row[~pd.isna(row)]
            cat, count = max_frequency_string(row)
            most_sel.append(cat)
            count_sel.append(count)
        most_sel_list.append(np.array(most_sel))
        count_sel_list.append(np.array(count_sel))
        
        # how many times correct option selected across 10 runs
        comparison_mat.append(comparison.sum(axis=1).values) 
        
        # accuracy for each run
        bm_across_models.append(comparison.sum(axis=0).values) 
        
    most_sel_list_and_t.append(most_sel_list)
    count_sel_list_and_t.append(count_sel_list)
    comparison_mat_all.append(np.array(comparison_mat))
    bm_across_models_and_t.append(bm_across_models)
    
comparison_mat = np.array(comparison_mat_all)
most_sel_list_and_t = np.array(most_sel_list_and_t)
count_sel_list_and_t = np.array(count_sel_list_and_t)

indexes_questions = rep_df.index    

In [ ]:
print(comparison_mat.shape) # accuracy (10/10 = 100%, 0/10 = 0%)
print(most_sel_list_and_t.shape) # majority vote
print(count_sel_list_and_t.shape) # frequency of majority vote

Saving the majority vote frequency:

In [ ]:
unique_vals = np.unique(most_sel_list_and_t)
count_t_m = np.zeros((len(temperature_list), len(model_list), len(unique_vals)), dtype=int)
list_df = []
for id_t, t in enumerate(temperature_list):
    for id_m, m in enumerate(model_list):
        for id_v, v_ in enumerate(unique_vals):
            count_t_m[id_t, id_m, id_v] = np.sum(most_sel_list_and_t[id_t, id_m] == v_)
    majority_vote_ = pd.DataFrame(index=model_list, columns=unique_vals, data=count_t_m[id_t])
    majority_vote_.to_latex(f"tables/majority_vote_T={t}.tex")

In [ ]:
majority_vote_

Below, dataframes one per temperature, containing number of correct responses across runs (row) and models (column):

In [ ]:
# this is a list of dataframes, one per temperature
df_accuracy_runs_list = [pd.DataFrame(bm_models_, index=model_list, columns=np.arange(1,11)).T 
                         for bm_models_ in bm_across_models_and_t]

In [ ]:
df_accuracy_runs_list[0]

In [ ]:
# Report hallucination for each model

counting_outputs = []
index = []

for t_ in temperature_list:
    for m_ in model_file: 
        print(t_, m_)
        filepath = f"{path_repetitions}/T={t_}/{m_}/{m_}_promptID_001_repetitions.xlsx"
        dct_counter = evaluate_response_distrib(filepath)
        print("HERE")
        index.append(f"T={t_} m={m_}")
        counting_outputs.append([dct_counter[k] for k in sorted(dct_counter.keys())])
columns_name = sorted(dct_counter.keys())

df_counter = pd.DataFrame(data=counting_outputs, 
                          index=index,
                          columns=columns_name)
df_counter.to_latex("tables/readability_experiment2_20-04-2026.tex")
df_counter

In [ ]:
def plot_singular_responses(matrix_comparison, 
                            across_models=True, 
                            temp_id=1, 
                            model_id=None,
                            saveplot=False,
                            plotname=None):
    # temp 0 - 0.3, temp 1 - 0.6, temp 2 - 1.
    colors_palette = [
    "#b2182b",  # 0 - deep red
    "#d6604d",  # 1 - red-orange
    "#f4a582",  # 2 - orange
    "#fddbc7",  # 3 - light orange
    "#fee8c8",  # 4 - beige
    "#f7f7f7",  # 5 - white/neutral
    "#d1e5f0",  # 6 - very light blue
    "#92c5de",  # 7 - light blue
    "#4393c3",  # 8 - medium blue
    "#2166ac",  # 9 - dark blue
    "#053061",  # 10 - deepest blue
    ]
    
    fig, ax = plt.subplots(figsize=(30, 7))
    ax.spines['bottom'].set_linewidth(2)
    ax.spines['left'].set_linewidth(2)
    ax.spines['top'].set_linewidth(2)
    ax.spines['right'].set_linewidth(2)
    ax.tick_params(width=2)
    
    if across_models: 
        sorted_id_answer = np.argsort(comparison_mat[temp_id][0]) # temperature 0.6, sorted by Huatuo
        matrix_to_iter = comparison_mat[temp_id]
        ylabel = model_list
        plt.title(f"LLMs at T={temperature_list[temp_id]}", fontsize=40, y=1.4)
        
    else: 
        sorted_id_answer = np.argsort(comparison_mat[0, model_id]) # temperature 0.6, sorted by Huatuo
        matrix_to_iter = comparison_mat[:, model_id]
        ylabel = temperature_list
        ax.set_ylabel("Temperature", fontsize=25)
        plt.title(f"{model_list[model_id]} at Different Temperatures", fontsize=40, y=1.4)

    n_bars = matrix_to_iter.shape[0]

    for i_iterator, votes_model in enumerate(matrix_to_iter):
        votes_model = votes_model[sorted_id_answer]
        for j, val in enumerate(votes_model):
            ax.barh(i_iterator, width=1, left=j, color=colors_palette[val], edgecolor='black')
            
    sorted_labs = meta_labs_tab_df['labs'].values[sorted_id_answer]
    for i_counter, l_ in enumerate(sorted_labs):
        if l_:
            color = 'white'
        else:
            color = 'lightgray'
        ax.barh(matrix_to_iter.shape[0], width=1, left=i_counter, color=color, edgecolor='black')

    # Set y-axis labels
    ax.set_yticks(range(n_bars + 1))
    ax.set_yticklabels(ylabel + ["Lab Tests"], rotation=0, fontsize=12)

    # Add legend for shades
    from matplotlib.patches import Patch
    
    text_legend = list(np.arange(11).astype(str)) + ['Yes', 'No']
    colors_palette_all = colors_palette + ['white', 'lightgray']
    legend_labels = [Patch(facecolor=colors_palette_all[i], label=f'{val}', edgecolor='black') 
                     for i, val in enumerate(text_legend)]
    
    ax.legend(handles=legend_labels, title='# Times Correct Option is Selected', 
              bbox_to_anchor=(0.5, 1.4), loc='upper center',
              ncol=13, fontsize=25, title_fontsize=25
              )

    ax.set_xticks(np.arange(len(sorted_id_answer))+0.5, 
                  indexes_questions[sorted_id_answer]);

    
    tick = ax.yaxis.get_major_ticks()
    for t in tick:
        t.label1.set_fontsize(25) 

    tick = ax.xaxis.get_major_ticks()
    for t in tick:
        t.label1.set_fontsize(13) 

    plt.xlabel("Case ID", fontsize=30)
    
    plt.tight_layout()
    if saveplot:
        plt.savefig(plotname)
    plt.show()

In [ ]:
for id_t, t_ in enumerate(temperature_list):
    plotname = f"plots/responses_exp2_extensive_T={t_}.pdf"

    plot_singular_responses(comparison_mat, 
                            across_models=True, 
                            temp_id=id_t, 
                            model_id=None,
                            saveplot=True,
                            plotname=plotname)

In [ ]:
for id_m, m_ in enumerate(model_list):
    plotname = f"plots/responses_exp2_extensive_m={m_}.pdf"

    plot_singular_responses(comparison_mat, across_models=False, model_id=id_m,
                            saveplot=True, plotname=plotname)

In [ ]:
# Compact summaries used for manuscript checks.
temperature_comparison = pd.DataFrame({
    "Huatuo T=0.6 > T=0.3": [(comparison_mat[1, 0] > comparison_mat[0, 0]).sum()],
    "Huatuo T=1.0 > T=0.3": [(comparison_mat[2, 0] > comparison_mat[0, 0]).sum()],
    "Huatuo T=1.0 > T=0.6": [(comparison_mat[2, 0] > comparison_mat[1, 0]).sum()],
    "Huatuo T=0.3 > T=1.0": [(comparison_mat[0, 0] > comparison_mat[2, 0]).sum()],
})

dominance_rows = []
for temp_idx, temperature in enumerate(temperature_list):
    for model_idx, model_name in enumerate(model_list):
        bm_best_or_tied = comparison_mat[temp_idx, model_idx] >= comparison_mat[temp_idx].max(axis=0)
        dominance_rows.append({
            "temperature": temperature,
            "model": model_name,
            "best_or_tied_cases": int(bm_best_or_tied.sum()),
        })

dominance_summary = pd.DataFrame(dominance_rows)
temperature_comparison, dominance_summary


In [ ]:
# Distribution of the number of correct selections across 10 repetitions at T=0.3.
vals = pd.DataFrame(comparison_mat[0], index=model_list, columns=df_all['truth'].index)
possible_values = list(range(0, 11))
counts_per_row = vals.apply(
    lambda row: row.value_counts().reindex(possible_values, fill_value=0),
    axis=1,
).T
counts_per_col = counts_per_row.T
never_correct = (vals == 0).sum(axis=1)

counts_per_row, counts_per_col, never_correct


In [ ]:
correct_consistent_tables = {}
for id_t, t_ in enumerate(temperature_list):
    result = np.zeros((len(model_file), 2), dtype=int)
    for id_m, m_ in enumerate(model_file):
        n_correct_10 = np.where(comparison_mat[id_t, id_m] == 10)[0].size
        n_consistent_10 = np.where(count_sel_list_and_t[id_t, id_m] == 10)[0].size
        result[id_m, :] = [n_correct_10, n_consistent_10 - n_correct_10]

    correct_consistent_tables[t_] = pd.DataFrame(
        result,
        index=model_list,
        columns=["# correct 10 times", "# consistently incorrect 10 times"],
    )

correct_consistent_tables


# Heatmaps and representations


# Answers Correctness Across Runs at Different Temperatures

Number of correct responses across each run.


In [ ]:
# Consistency and accuracy across runs at different temperatures.
fig, ax = plt.subplots(figsize=(32, 12), nrows=3, ncols=6, sharex=True, sharey=True)
row_colors = [(0.3060, 0.4285, 0.7891), "lightsteelblue", "gold"]
model_list_plot = [
    "HuatuoGPT-o1-8B",
    "Meditron3-8B",
    "Llama3.1-8B-I",
    "Diabetica-o1",
    "Diabetica-7B",
    "Qwen2-7B-I",
]

global_max = 0
for i_t, _ in enumerate(temperature_list):
    for i_m in range(len(model_list)):
        data_heatmap = np.zeros((8, 11))
        bm_majority = ~np.isnan(count_sel_list_and_t[i_t, i_m])
        for consistency, correctness in zip(
            count_sel_list_and_t[i_t, i_m, bm_majority],
            comparison_mat[i_t, i_m, bm_majority],
        ):
            data_heatmap[int(consistency) - 3, int(correctness)] += 1
        data_heatmap = np.vstack((data_heatmap[:5, :].sum(axis=0).reshape(1, -1), data_heatmap[5:, :]))
        global_max = max(global_max, np.max(data_heatmap))

for i_t, t in enumerate(temperature_list):
    for i_m in range(len(model_list)):
        data_heatmap = np.zeros((8, 11))
        bm_majority = ~np.isnan(count_sel_list_and_t[i_t, i_m])
        for consistency, correctness in zip(
            count_sel_list_and_t[i_t, i_m, bm_majority],
            comparison_mat[i_t, i_m, bm_majority],
        ):
            data_heatmap[int(consistency) - 3, int(correctness)] += 1

        data_heatmap = np.vstack((data_heatmap[:5, :].sum(axis=0).reshape(1, -1), data_heatmap[5:, :]))[::-1]
        data = pd.DataFrame(data_heatmap, columns=range(11), index=[1.0, 0.9, 0.8, r"$\leq 0.7$"])

        for spine in ax[i_t, i_m].spines.values():
            spine.set_linewidth(4)
            spine.set_color(row_colors[i_t])
        ax[i_t, i_m].tick_params(width=2)
        ax[i_t, i_m].grid(True, color="lightgray")

        x_pos, y_pos, sizes, texts = [], [], [], []
        for y_idx, _ in enumerate(data.index):
            for x_idx, _ in enumerate(data.columns):
                val = int(data.iloc[y_idx, x_idx])
                if val > 0:
                    x_pos.append(x_idx)
                    y_pos.append(y_idx)
                    sizes.append(val * 100)
                    texts.append(str(val))

        ax[i_t, i_m].scatter(
            x_pos,
            y_pos,
            s=sizes,
            c=[int(t) for t in texts],
            cmap="RdPu",
            edgecolors="none",
            marker="o",
            zorder=4,
            vmin=0,
            vmax=global_max,
        )

        ax[i_t, i_m].margins(x=0.3, y=0.35)
        ax[i_t, i_m].set_yticks(range(len(data.index)))
        ax[i_t, i_m].set_yticklabels(data.index, fontsize=28)
        ax[i_t, i_m].set_title(model_list_plot[i_m] if i_t == 0 else "", fontsize=30, weight="bold")
        ax[i_t, i_m].set_xlim([-1, 11])
        ticks = ax[i_t, i_m].get_xticks()
        ax[i_t, i_m].set_xticklabels([f"{tick / 10:.1f}" for tick in ticks], fontsize=20)

    ax[i_t, 0].set_ylabel(f"T={t}", fontsize=30, weight="bold", color=row_colors[i_t])

fig.supxlabel(r"$f{Mean\ Accuracy\ Across\ 10\ Runs}$", fontsize=35)
fig.supylabel(r"$f{Consistency\ of\ Majority\ Vote}$", fontsize=35, x=0.05)
ax[0, 0].invert_yaxis()

sm = cm.ScalarMappable(cmap="RdPu", norm=plt.Normalize(vmin=0, vmax=global_max))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax.ravel().tolist(), orientation="vertical", fraction=0.02, pad=0.008)
cbar.set_label("# questions", fontsize=35)
for spine in cbar.ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(2)
cbar.ax.tick_params(labelsize=30, width=2)

plt.savefig("plots/answer_correctness_consistency.pdf")
plt.show()
